In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# ==========================================================
# MODEL PATH
# ==========================================================

CHECKPOINT =  "./V5A_Final_Merged_Model"

# ==========================================================
# LOAD MODEL
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 100)
print("LOADING V5A MODEL")
print("=" * 100)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

print("Loading merged model...")
model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()

print("✅ V5A Model Loaded Successfully")


# ==========================================================
# SYSTEM PROMPT
# ==========================================================

SYSTEM_PROMPT = """You are Compatifi V5A.

Your task is to determine what the assistant should try to accomplish
in the current conversation.

V5A predicts the assistant's immediate objective.

It does NOT predict:
- the user's long-term goals
- memories
- personality
- emotions as memories
- a response
- future events

Use ONLY:
- Domain
- Relationship
- Conversation

Always predict ONE primary objective.

The secondary objective may be "None" if unnecessary.

Valid primary objectives include:
- Emotional Support
- Reduce Anxiety
- Solve Problem
- Decision Support
- Planning Assistance
- Motivation
- Information Sharing
- Maintain Rapport

Valid secondary objectives include:
- None
- Build Confidence
- Clarify Situation
- Encourage Reflection
- Suggest Next Steps
- Maintain Rapport

Priority must be:
- High
- Medium
- Low

Return ONLY valid JSON.

Required format:

{
  "primary_objective": "...",
  "secondary_objective": "...",
  "priority": "...",
  "reason": "..."
}

The reason must be based ONLY on information explicitly present
in the conversation.

Do not generate a reply to the user.
"""



/home/ubuntu/V5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOADING V5A MODEL
Loading tokenizer...
Loading merged model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:10<00:00,  2.59s/it]

✅ V5A Model Loaded Successfully


In [2]:

# ==========================================================
# TEST CONVERSATION
# ==========================================================



test_conversations = [

    {
        "name": "SECONDARY OBJECTIVE TEST - Interview Anxiety",
        "domain": "career",
        "relationship": "Mentor",

        "conversation": """
Mentor: How are you feeling about your interview tomorrow?

User: Honestly, I'm really nervous. I know I've prepared well,
but I keep thinking I'm going to fail.

Mentor: What makes you feel that way?

User: I keep doubting whether I'm good enough for the position.

Mentor: Your preparation has been strong, and you've already
practiced the difficult questions several times.

User: I know, but I still don't feel confident.

Mentor: Then let's focus on reminding you of what you've already
accomplished and preparing you to approach the interview calmly.

User: Yes, I think I need that.
"""
    }

]



In [3]:

# ==========================================================
# RUN TESTS
# ==========================================================

for test in test_conversations:

    print("\n")
    print("=" * 100)
    print(test["name"])
    print("=" * 100)

    user_prompt = f"""
Domain: {test['domain']}

Relationship: {test['relationship']}

Conversation:

{test['conversation']}

Instruction:
Predict the assistant objective.
"""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    # ------------------------------------------------------
    # CHAT TEMPLATE
    # ------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # ------------------------------------------------------
    # TOKENIZE
    # ------------------------------------------------------

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    # ------------------------------------------------------
    # GENERATE
    # ------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    # ------------------------------------------------------
    # DECODE ONLY NEW TOKENS
    # ------------------------------------------------------

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    # ------------------------------------------------------
    # PRINT RESULT
    # ------------------------------------------------------

    print("\nMODEL OUTPUT")
    print("-" * 100)
    print(response)

    print("\n")


/home/ubuntu/V5/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/ubuntu/V5/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/ubuntu/V5/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(




SECONDARY OBJECTIVE TEST - Interview Anxiety

MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

{"primary_objective":"Reduce Anxiety","secondary_objective":"Increase Confidence","priority":"High","reason":"The user is feeling very nervous and doubting their readiness for an upcoming interview.","memories":null,"conversation":null}




In [1]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ==========================================================
# MODEL PATH
# ==========================================================

CHECKPOINT =  r"E:\Compatifi Model\V5A_Final_Merged_Model"

# ==========================================================
# LOAD MODEL
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 100)
print("LOADING V5A MODEL")
print("=" * 100)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT,
    local_files_only=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Loading merged model...")

model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True,
)

model.eval()

print("✅ V5A Model Loaded Successfully")


# ==========================================================
# SYSTEM PROMPT
# ==========================================================

SYSTEM_PROMPT = """You are Compatifi V5A.

Your ONLY task is to predict the immediate objective
that the assistant should accomplish in the current conversation.

Do NOT generate an assistant reply.

Do NOT extract memories.
Do NOT extract people memories.
Do NOT generate conversation summaries.
Do NOT generate personality information.
Do NOT generate user long-term goals.

Return EXACTLY ONE JSON object.

The JSON object MUST contain EXACTLY these four fields:

{
  "primary_objective": "...",
  "secondary_objective": "...",
  "priority": "...",
  "reason": "..."
}

Valid primary objectives:

- Emotional Support
- Reduce Anxiety
- Solve Problem
- Decision Support
- Planning Assistance
- Motivation
- Information Sharing
- Maintain Rapport

Valid secondary objectives:

- None
- Increase Confidence
- Clarify Situation
- Encourage Reflection
- Suggest Next Steps
- Maintain Rapport

Valid priorities:

- High
- Medium
- Low

Rules:

1. Output exactly four fields.
2. The four fields must be:
   primary_objective
   secondary_objective
   priority
   reason

3. Never output:
   memories
   people
   conversation
   personality
   user_goal
   or any other fields.

4. primary_objective must describe what the assistant
   should accomplish immediately.

5. secondary_objective must support the primary objective.

6. reason must be based ONLY on information explicitly
   present in the conversation.

7. Do not generate a reply to the user.

8. Return JSON only.

9. Do not output markdown.

10. Do not output <think>.

11. Do not output text before or after the JSON object.

The output MUST begin with { and end with }.
"""


# ==========================================================
# TEST CONVERSATIONS
# ==========================================================

test_conversations = [

    {
        "name": "SECONDARY OBJECTIVE TEST - Interview Anxiety",

        "domain": "career",

        "relationship": "Mentor",

        "conversation": """
Mentor: How are you feeling about your interview tomorrow?

User: Honestly, I'm really nervous. I know I've prepared well,
but I keep thinking I'm going to fail.

Mentor: What makes you feel that way?

User: I keep doubting whether I'm good enough for the position.

Mentor: Your preparation has been strong, and you've already
practiced the difficult questions several times.

User: I know, but I still don't feel confident.

Mentor: Then let's focus on reminding you of what you've already
accomplished and preparing you to approach the interview calmly.

User: Yes, I think I need that.
"""
    }

]


# ==========================================================
# RUN TESTS
# ==========================================================

for test in test_conversations:

    print("\n")
    print("=" * 100)
    print(test["name"])
    print("=" * 100)

    # ------------------------------------------------------
    # INPUT
    # ------------------------------------------------------

    user_prompt = f"""Domain: {test['domain']}

Relationship: {test['relationship']}

Conversation:
{test['conversation']}

Instruction:
Predict the assistant objective.
"""

    # ------------------------------------------------------
    # CHAT MESSAGES
    # ------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    # ------------------------------------------------------
    # CHAT TEMPLATE
    # ------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # ------------------------------------------------------
    # TOKENIZE
    # ------------------------------------------------------

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    # ------------------------------------------------------
    # GENERATE
    # ------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # ------------------------------------------------------
    # DECODE ONLY GENERATED TOKENS
    # ------------------------------------------------------

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # ------------------------------------------------------
    # REMOVE THINK BLOCK
    # ------------------------------------------------------

    if "<think>" in response:

        if "</think>" in response:
            response = response.split("</think>", 1)[1].strip()

        else:
            response = response.replace("<think>", "").strip()

    # ------------------------------------------------------
    # FIND JSON
    # ------------------------------------------------------

    start = response.find("{")
    end = response.rfind("}")

    if start != -1 and end != -1 and end > start:

        json_text = response[start:end + 1]

    else:

        json_text = response

    # ------------------------------------------------------
    # PARSE JSON
    # ------------------------------------------------------

    try:

        result = json.loads(json_text)

        # --------------------------------------------------
        # KEEP ONLY V5A FOUR FIELDS
        # --------------------------------------------------

        v5a_result = {
            "primary_objective": result.get(
                "primary_objective"
            ),

            "secondary_objective": result.get(
                "secondary_objective"
            ),

            "priority": result.get(
                "priority"
            ),

            "reason": result.get(
                "reason"
            ),
        }

        # --------------------------------------------------
        # PRINT FINAL V5A OUTPUT
        # --------------------------------------------------

        print("\nMODEL OUTPUT")
        print("-" * 100)

        print(
            json.dumps(
                v5a_result,
                indent=2,
                ensure_ascii=False,
            )
        )

    except json.JSONDecodeError:

        print("\n❌ INVALID JSON FROM MODEL")
        print("-" * 100)
        print(response)

    print("\n")

g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOADING V5A MODEL
Loading tokenizer...
Loading merged model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:51<00:00, 12.84s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ V5A Model Loaded Successfully


SECONDARY OBJECTIVE TEST - Interview Anxiety

MODEL OUTPUT
----------------------------------------------------------------------------------------------------
{
  "primary_objective": "Reduce Anxiety",
  "secondary_objective": "Increase Confidence",
  "priority": "High",
  "reason": "The user is feeling very nervous and doubting their readiness for an upcoming interview."
}


